# Lab 1 – Implementing Distributed Tracing

**Estimated Time:** 90-120 minutes  
**Difficulty:** Intermediate

## Learning Objectives
- Configure Langfuse for end-to-end trace collection in LLM applications
- Instrument Retrieval-Augmented Generation (RAG) pipelines with hierarchical spans
- Attribute latency, tokens, and cost to user journeys
- Build custom dashboards, alerts, and OpenTelemetry exporters for Langfuse
- Apply sampling, anomaly detection, and performance diagnostics at scale

## Prerequisites & Setup
- Python 3.10+ environment with `langfuse`, `openai`, `opentelemetry-sdk`, `pandas`, `numpy`, `httpx`, and `tqdm` installed
- Environment variables set: `LANGFUSE_API_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_HOST`, `OPENAI_API_KEY`
- Langfuse project created (self-hosted or cloud) with API keys copied
- Optional: Prometheus & Grafana stack for validating exported metrics

> **Tip:** Create a `.env` file and use `python-dotenv` if you prefer managing secrets locally during the lab.

## Lab Structure
This lab contains seven exercises that build on one another. Complete them in order and validate each milestone before moving on.

1. Langfuse instrumentation bootstrap
2. Instrumenting a multi-step RAG pipeline
3. Cost attribution and token accounting
4. Performance diagnostics and bottleneck analysis
5. Custom metrics and dashboard export
6. Alerting on anomalies and error bursts
7. OpenTelemetry integration for interoperable tracing

Each exercise includes starter code annotated with `TODO` blocks. Fill in the logic, run the cell, and jot down observations in the provided reflection cells.

## Exercise 1: Bootstrap Langfuse Instrumentation
**Scenario:** Your team needs a reliable tracing client that tags every request with environment metadata.

**Success Criteria**
- Langfuse client authenticates with keys from environment variables
- `TraceContext` dataclass captures request metadata (user, feature flag, release)
- Helper returns a ready-to-use Langfuse trace with a root span

**Hints**
- Use `os.getenv` to read credentials
- Fail fast if any required secret is missing
- Provide helper methods for span tagging to reuse later

In [ ]:
import os
import uuid
from dataclasses import dataclass, field
from typing import Dict, Optional
from langfuse import Langfuse
from langfuse.client import LangfuseClient

@dataclass
class TraceContext:
    """Container for request-scoped metadata."""
    user_id: str
    feature: str
    environment: str = "development"
    release: Optional[str] = None
    attributes: Dict[str, str] = field(default_factory=dict)

    def to_tags(self) -> Dict[str, str]:
        base = {
            "user_id": self.user_id,
            "feature": self.feature,
            "environment": self.environment,
        }
        if self.release:
            base["release"] = self.release
        base.update(self.attributes)
        return base

def bootstrap_langfuse_client() -> LangfuseClient:
    """Create an authenticated Langfuse client with defensive checks."""
    api_key = os.getenv("LANGFUSE_API_KEY")
    public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
    host = os.getenv("LANGFUSE_HOST")

    # TODO: Validate that api_key, public_key, and host are set; raise a descriptive error if any are missing
    # TODO: Instantiate and return the Langfuse client configured with retries and timeouts suitable for production

def start_trace(client: LangfuseClient, context: TraceContext):
    """Return a trace handle with a root span tagged using the provided context."""
    trace_id = str(uuid.uuid4())
    # TODO: Create a Langfuse trace (consider using client.trace) and attach context metadata
    # TODO: Return both the trace and a root span so downstream steps can create children

# Smoke-test scaffolding (uncomment after implementing)
# if __name__ == "__main__":
#     client = bootstrap_langfuse_client()
#     trace, root_span = start_trace(client, TraceContext(user_id="user-123", feature="rag-search"))
#     print("Trace initialized", trace.id)

**Reflection & Verification**
- [ ] Credentials load securely without hardcoding
- [ ] Root span includes user, feature, environment tags
- [ ] Failure paths emit actionable error messages

## Exercise 2: Instrument a Multi-Step RAG Pipeline
**Scenario:** Add span-level visibility to a Retrieval-Augmented Generation flow.

**Success Criteria**
- Async RAG pipeline creates spans for embed → retrieve → rerank → generate
- Each span records duration and key attributes (vector store, top-k, model)
- Trace is flushed even if an exception occurs

**Hints**
- Use `asyncio` and `httpx.AsyncClient` to simulate remote calls
- Wrap each logical step in `async with span.start_child(...)`
- Capture inputs/outputs via `span.update` while redacting sensitive fields

In [ ]:
import asyncio
import httpx
from typing import List, Tuple

async def embed_query(query: str) -> List[float]:
    # TODO: Record a child span named "embed" with model metadata and latency
    await asyncio.sleep(0.05)  # Simulated latency
    return [0.1, 0.2, 0.3]

async def retrieve_documents(embedding: List[float]) -> List[Dict]:
    # TODO: Instrument this function with a "retrieve" span including vector store details
    await asyncio.sleep(0.08)
    return [{"document_id": "doc-1", "score": 0.87}]

async def rerank_results(results: List[Dict]) -> List[Dict]:
    # TODO: Add a span capturing reranker configuration (model name, top_n)
    await asyncio.sleep(0.03)
    return results

async def generate_answer(query: str, context: List[Dict]) -> str:
    # TODO: Create a "generate" span, attach token usage, and capture the final response
    await asyncio.sleep(0.12)
    return "RAG response placeholder"

async def rag_pipeline(prompt: str) -> str:
    # TODO: Obtain the Langfuse trace and root span from Exercise 1
    # TODO: Ensure spans are closed properly using async context managers
    embedding = await embed_query(prompt)
    results = await retrieve_documents(embedding)
    reranked = await rerank_results(results)
    answer = await generate_answer(prompt, reranked)
    # TODO: Record completion status on the root span
    return answer

# TODO: Wire up a main() that runs rag_pipeline and flushes the trace on completion

**Reflection & Verification**
- [ ] Each logical step emits a span visible in Langfuse
- [ ] Trace continues when downstream calls succeed
- [ ] Exceptions surface in the trace with meaningful tags

## Exercise 3: Attribute Tokens and Cost
**Scenario:** Finance needs per-user cost attribution for model usage.

**Success Criteria**
- Token counts extracted from OpenAI responses (prompt + completion)
- Cost computed via `PRICING` dictionary and attached to spans
- Aggregate cost per trace surfaced for dashboarding

**Hints**
- Store costs using span attributes (e.g., `span.update(cost=...)`)
- Use `Decimal` for accurate currency math
- Consider tagging spans with `billing_account` or `tenant_id`

In [ ]:
from decimal import Decimal
from typing import Any

PRICING = {
    "gpt-4o-mini": {"prompt": Decimal(0.0000005), "completion": Decimal(0.0000015)},
    "text-embedding-3-small": {"prompt": Decimal(0.0000001), "completion": Decimal(0)}
}

def calculate_cost(model: str, prompt_tokens: int, completion_tokens: int) -> Decimal:
    # TODO: Lookup pricing, multiply by tokens, and return a Decimal cost
    ...

def record_span_cost(span: Any, model: str, token_usage: Dict[str, int]) -> None:
    """Attach cost metadata to a Langfuse span."""
    # TODO: Compute cost using calculate_cost
    # TODO: Update span with cost, tokens, and billing tags
    # TODO: Return the cost so the caller can aggregate at the trace level

def aggregate_trace_cost(spans: List[Any]) -> Decimal:
    # TODO: Traverse spans, sum their attached cost attribute, and return the total
    ...

# Example usage inside generate_answer (hook this into Exercise 2 once implemented)
# token_usage = {"prompt_tokens": 1234, "completion_tokens": 567}
# cost = record_span_cost(current_span, model="gpt-4o-mini", token_usage=token_usage)
# print("Cost for span", cost)

**Reflection & Verification**
- [ ] Costs appear alongside spans in Langfuse
- [ ] Totals reconcile with OpenAI usage dashboards
- [ ] No floating-point rounding errors in currency calculations

## Exercise 4: Diagnose Performance Bottlenecks
**Scenario:** SRE needs automated reports highlighting slow spans and percentile latencies.

**Success Criteria**
- Helper computes P50/P95/P99 latencies per span type using pandas or numpy
- Functions flag spans exceeding SLO thresholds
- Root cause report identifies the slowest step and suggests remediation

**Hints**
- Use `numpy.percentile` for latency quantiles
- Store raw span data in a DataFrame with columns (name, duration_ms, status)
- Output structured dicts so dashboards can consume them later

In [ ]:
import numpy as np
import pandas as pd
from typing import Iterable

def build_span_dataframe(spans: Iterable[Any]) -> pd.DataFrame:
    data = []
    for span in spans:
        # TODO: Extract name, duration_ms, status, and custom attributes
        data.append({
            "name": span.name,
            "duration_ms": span.duration_ms,
            "status": span.status,
            # TODO: Pull additional metadata such as model or vector store
        })
    return pd.DataFrame(data)

def percentile_report(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: Group by span name and compute P50/P95/P99
    ...

def detect_slo_breaches(df: pd.DataFrame, slo_targets: Dict[str, float]) -> pd.DataFrame:
    # TODO: Compare actual durations against slo_targets per span name
    ...

def summarize_bottlenecks(df: pd.DataFrame) -> Dict[str, Any]:
    # TODO: Identify the slowest span and craft a remediation suggestion string
    ...

# TODO: Demonstrate usage with mock data once the helpers are implemented

**Reflection & Verification**
- [ ] Performance summary highlights true bottlenecks
- [ ] SLO breaches match expectations from traces
- [ ] Reports are ready for dashboard export (CSV or JSON)

## Exercise 5: Build Custom Metrics & Dashboards
**Scenario:** Product analytics wants rolled-up metrics for daily dashboards.

**Success Criteria**
- Span data aggregated into daily metrics (requests, median latency, cost)
- Functions output dictionaries ready to push into Langfuse custom metrics or BI tools
- Capability to filter by feature, user segment, or model

**Hints**
- Use DataFrame grouping with `.resample('D')` or manual bucketing
- Consider returning both summary stats and raw series
- Ensure metrics align with those in Lesson 2 dashboards

In [ ]:
from datetime import datetime

def aggregate_metrics(df: pd.DataFrame, feature: Optional[str] = None) -> Dict[str, Any]:
    # TODO: Filter by feature if provided
    # TODO: Bucket spans by day and compute counts, median duration, total cost
    ...

def prepare_dashboard_payload(metrics: Dict[str, Any]) -> Dict[str, Any]:
    # TODO: Shape the data for Langfuse custom metrics or external dashboards
    ...

def export_metrics(metrics: Dict[str, Any], destination: str) -> None:
    # TODO: Implement a stub that would push metrics to a chosen sink (Langfuse, S3, etc.)
    ...

# TODO: Build a demo dataset and run through the aggregation pipeline

**Reflection & Verification**
- [ ] Metrics reconcile with raw trace counts
- [ ] Payload structure is consumable by dashboards
- [ ] Filters allow slicing by feature or user attribute

## Exercise 6: Alert on Anomalies
**Scenario:** Detect sudden latency spikes or error bursts before users complain.

**Success Criteria**
- Z-score based anomaly detector for latency and error rate
- Alert routing configuration that differentiates warning vs critical
- Sample notebook output demonstrating an alert firing

**Hints**
- Maintain a rolling window of metrics for z-score calculation
- Use dataclass `Alert` with severity, message, metadata
- Integrate with Langfuse scores or annotations for long-term tracking

In [ ]:
from dataclasses import dataclass

@dataclass
class Alert:
    severity: str
    message: str
    metadata: Dict[str, Any]

def compute_z_score(values: List[float], new_value: float) -> float:
    # TODO: Implement mean/stddev based z-score (handle zero variance)
    ...

def detect_latency_anomaly(latencies: List[float], latest_latency: float, threshold: float = 2.5) -> Optional[Alert]:
    # TODO: Compute z-score and return an Alert if threshold exceeded
    ...

def detect_error_spike(error_counts: List[int], latest_count: int, baseline: float) -> Optional[Alert]:
    # TODO: Compare latest_count against baseline (e.g., rolling mean) and raise alert
    ...

def route_alert(alert: Alert) -> None:
    # TODO: Stub routing logic (print, webhook, Langfuse annotation)
    ...

# TODO: Simulate a latency spike to verify alerts work as expected

**Reflection & Verification**
- [ ] Alerts fire for genuine anomalies and remain quiet otherwise
- [ ] Alert payload contains span identifiers for debugging
- [ ] Routing logic integrates with existing on-call tooling

## Exercise 7: Integrate OpenTelemetry Exporter
**Scenario:** Unify tracing data across teams by streaming Langfuse spans into OpenTelemetry.

**Success Criteria**
- Custom exporter bridges Langfuse spans to OpenTelemetry SDK
- Sampling strategy skips low-value traces while preserving errors
- Flush logic handles graceful shutdown and backpressure

**Hints**
- Subclass `SpanExporter` from `opentelemetry.sdk.trace.export`
- Translate Langfuse span fields to OTLP attributes
- Use `BatchSpanProcessor` for efficiency

In [ ]:
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult
from opentelemetry.sdk.trace import ReadableSpan

class LangfuseSpanExporter(SpanExporter):
    """Bridge OpenTelemetry spans into Langfuse."""
    def __init__(self, client: LangfuseClient, sample_rate: float = 1.0):
        self.client = client
        self.sample_rate = sample_rate

    def export(self, spans: List[ReadableSpan]) -> SpanExportResult:
        # TODO: Iterate over spans, apply sampling, and push to Langfuse
        # TODO: Map OTLP attributes to Langfuse fields (name, input, output, tags)
        ...

    def shutdown(self) -> None:
        # TODO: Flush any buffered spans and close resources
        ...

def configure_opentelemetry_exporter(client: LangfuseClient) -> None:
    # TODO: Instantiate LangfuseSpanExporter, wrap with BatchSpanProcessor, and attach to tracer provider
    ...

# TODO: Demonstrate how to emit a sample OTEL span and confirm it arrives in Langfuse

**Reflection & Verification**
- [ ] OTEL spans appear in Langfuse with preserved hierarchy
- [ ] Sampling avoids overwhelming the backend
- [ ] Shutdown paths flush data without loss

## Wrap-Up
You instrumented an end-to-end RAG workflow, enriched spans with cost and performance data, and built operational guardrails for distributed tracing. Continue by:
- Rolling out the tracing package to staging and production environments
- Wiring alerts into on-call rotation channels
- Automating dashboard exports for weekly operations reviews
- Comparing Langfuse traces with existing observability stacks (Datadog, Honeycomb, etc.)

## Submission Checklist
- [ ] All TODOs in code cells resolved and executed without errors
- [ ] Screenshots or trace URLs demonstrating instrumentation in Langfuse
- [ ] Summary of performance findings and alerts triggered
- [ ] Notes on integration challenges and mitigation strategies